In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Jawaharlal Nehru Stadium, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,182.48,277.04,33.70,60.49,59.60,59.58,8.17,1.49,27.90,1.89,7.51,76.31,1.36,172.05,988.76,14.42,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,162.66,255.04,34.49,55.19,56.47,76.69,7.60,1.55,28.41,1.77,5.02,76.91,1.16,217.16,991.33,14.52,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,229.83,359.04,99.83,82.19,124.85,91.96,9.54,3.02,21.54,3.60,12.60,76.94,1.30,194.06,990.55,15.40,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,249.54,337.08,76.79,86.20,108.09,98.37,5.88,3.23,25.19,4.47,16.10,79.72,1.15,254.76,988.42,15.57,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,155.92,247.50,25.61,56.78,51.07,75.47,9.66,1.51,19.73,1.75,3.47,76.86,1.20,280.46,987.81,15.15,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,325.88,534.83,53.89,94.39,94.02,18.55,21.25,2.51,55.17,3.44,58.14,62.28,0.90,314.98,991.98,18.79,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,288.42,498.46,47.19,100.29,91.20,18.46,18.91,2.38,49.58,1.46,73.24,65.09,0.68,320.44,995.64,18.51,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,225.79,414.96,26.48,97.74,73.52,19.94,21.81,1.94,47.88,0.79,55.41,64.12,0.50,300.21,996.08,18.28,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,230.79,434.54,39.79,94.39,82.57,21.43,20.56,2.01,50.91,0.75,55.39,64.17,0.52,293.91,997.45,18.20,0.0,0.0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 1
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")

    


In [7]:


# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (319, 20)
          From Date           To Date   PM2.5    PM10     NO    NO2     NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  42.105  277.04  33.70  60.49   59.60   
1  02-01-2025 00:00  03-01-2025 00:00  42.105  255.04  34.49  55.19   56.47   
2  03-01-2025 00:00  04-01-2025 00:00  42.105  359.04  19.68  82.19   43.63   
3  04-01-2025 00:00  05-01-2025 00:00  42.105  337.08  76.79  86.20  108.09   
4  05-01-2025 00:00  06-01-2025 00:00  42.105  247.50  25.61  56.78   51.07   

     NH3   SO2    CO  Ozone  Benzene  Toluene     RH    WS      WD      BP  \
0  59.58  8.17  1.49  27.90     1.89     7.51  76.31  1.36  172.05  988.76   
1  34.69  7.60  1.55  28.41     1.77     5.02  76.91  1.16  217.16  991.33   
2  34.69  9.54  3.02  21.54     3.60    12.60  76.94  1.30  194.06  990.55   
3  34.69  5.88  3.23  25.19     4.47    16.10  79.72  1.15  254.76  988.42   
4  34.69  9.66  1.51  19.73     1.75     3.47  76.86  1.20  280.46  987.81   

      AT   RF  TOT-RF  
0  14.42 

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,-0.184155,1.523234,0.614092,0.168808,0.427198,2.248546,-0.614951,0.163372,-0.793216,0.514217,-0.438883,0.929901,0.428233,-1.324396,0.975933,-2.242266,-0.229794,-0.188982
1,02-01-2025 00:00,03-01-2025 00:00,-0.184155,1.245638,0.663075,-0.110171,0.289110,-0.125475,-0.810907,0.253239,-0.762276,0.419128,-0.656216,0.974228,0.002668,-0.320251,1.263042,-2.226146,-0.229794,-0.188982
2,03-01-2025 00:00,04-01-2025 00:00,-0.184155,2.557910,-0.255198,1.311041,-0.277357,-0.125475,-0.143969,2.454987,-1.179060,1.869224,0.005382,0.976444,0.300564,-0.834455,1.175904,-2.084287,-0.229794,-0.188982
3,04-01-2025 00:00,05-01-2025 00:00,-0.184155,2.280819,3.285826,1.522118,2.566451,-0.125475,-1.402214,2.769522,-0.957624,2.558613,0.310869,1.181826,-0.018610,0.516722,0.937950,-2.056883,-0.229794,-0.188982
4,05-01-2025 00:00,06-01-2025 00:00,-0.184155,1.150498,0.112483,-0.026477,0.050876,-0.125475,-0.102715,0.193327,-1.288867,0.403280,-0.791503,0.970534,0.087781,1.088801,0.869803,-2.124588,-0.229794,-0.188982
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,12-11-2025 00:00,13-11-2025 00:00,-0.184155,-0.093007,1.865944,1.953219,1.945719,-1.664916,-0.003018,1.691115,0.861177,1.742439,-0.088010,-0.106609,-0.550567,1.857213,1.335658,-1.537810,-0.229794,-0.188982
315,13-11-2025 00:00,14-11-2025 00:00,-0.184155,-0.093007,1.450520,2.263780,1.821308,-1.673501,-0.003018,1.496403,0.522048,0.173484,-0.088010,0.100988,-1.018688,1.978752,1.744537,-1.582946,-0.229794,-0.188982
316,14-11-2025 00:00,15-11-2025 00:00,-0.184155,-0.093007,0.166426,2.129554,1.041312,-1.532337,-0.003018,0.837376,0.418913,-0.357425,-0.088010,0.029327,-1.401697,1.528435,1.793692,-1.620023,-0.229794,-0.188982
317,15-11-2025 00:00,16-11-2025 00:00,-0.184155,-0.093007,0.991694,1.953219,1.440575,-1.390220,-0.003018,0.942221,0.602735,-0.389121,-0.088010,0.033021,-1.359140,1.388197,1.946743,-1.632919,-0.229794,-0.188982


In [10]:
df.to_excel('jawaharlalStadium2025.xlsx', index=False)